In [1]:
import duckdb
import pandas as pd

diag = duckdb.sql(
    "SELECT * FROM read_parquet('../data/gold/nyc/diagnosis.parquet')"
).df()

pd.set_option("display.max_colwidth", None)

print("=== A healthy listing ===")
print(diag[diag["issues_shown"] == 0]["diagnosis"].iloc[0])
print()
print("=== A listing with issues ===")
print(diag[diag["issues_shown"] == 3]["diagnosis"].iloc[0])

=== A healthy listing ===
Your listing looks healthy overall (score 100/100) — priced in line with similar listings, with a reasonably full calendar and no major warning signs.

=== A listing with issues ===
Health score: 35/100. Your price of $1,950 is far outside the normal range for similar listings — worth double-checking it's set correctly. Your price of $1,950 is higher than 100% of comparable listings nearby (median $188) — worth checking if that's intentional. Your calendar is 100% open in the next 30 days, versus 44% for similar listings — this could mean the price or listing needs a closer look.


In [2]:
#Check for any crashed sentences or missing numbers
# Search for any literal "nan" that slipped through despite the try/except
has_nan_text = diag["diagnosis"].str.contains("nan", case=False, na=False)
print("Diagnoses containing 'nan':", has_nan_text.sum())
if has_nan_text.sum() > 0:
    print(diag[has_nan_text][["listing_id", "diagnosis"]].head(5).to_string(index=False))

Diagnoses containing 'nan': 0


In [3]:
#Confirm the low-confidence caveat appears where expected
health = duckdb.sql(
    "SELECT listing_id, price_confidence FROM read_parquet('../data/gold/nyc/health_score.parquet')"
).df()
merged = diag.merge(health, on="listing_id")

low_conf = merged[merged["price_confidence"] == "low"]
print("Low-confidence listings:", len(low_conf))
print("...with the caveat prefix present:",
      low_conf["diagnosis"].str.startswith("Note:").sum())

Low-confidence listings: 61
...with the caveat prefix present: 61


In [4]:
#Read 5 random full diagnoses
sample = diag.sample(5, random_state=3)
for _, row in sample.iterrows():
    print(f"Listing {row.listing_id}: {row.diagnosis}")
    print()

Listing 563814027308620995: Health score: 45/100. Your price of $586 is higher than 100% of comparable listings nearby (median $167) — worth checking if that's intentional. Your calendar is 100% open in the next 30 days, versus 48% for similar listings — this could mean the price or listing needs a closer look. You have 53 reviews total, but none in the last 12 months — worth checking if something has changed with visibility or pricing.

Listing 39194327: Your listing looks healthy overall (score 100/100) — priced in line with similar listings, with a reasonably full calendar and no major warning signs.

Listing 1279026320984012410: Health score: 90/100. Your price of $167 is lower than 96% of comparable listings (median $442) — there may be room to raise it.

Listing 932578265767796981: Your listing looks healthy overall (score 100/100) — priced in line with similar listings, with a reasonably full calendar and no major warning signs.

Listing 1179802084308327475: Health score: 80/100

In [5]:
diag = duckdb.sql(
    "SELECT * FROM read_parquet('../data/gold/nyc/diagnosis.parquet')"
).df()

# Find a diagnosis that should now say "every comparable listing nearby"
matches = diag[diag["diagnosis"].str.contains("every comparable listing nearby", na=False)]
print("Diagnoses using the new phrasing:", len(matches))
print(matches["diagnosis"].iloc[0] if len(matches) else "none found")

Diagnoses using the new phrasing: 717
Health score: 80/100. Your price of $1,935 is higher than every comparable listing nearby (median $231) — worth checking if that's intentional.


In [6]:
low_conf_sample = duckdb.sql("""
    SELECT listing_id FROM read_parquet('../data/gold/nyc/health_score.parquet')
    WHERE price_confidence = 'low' LIMIT 3
""").df()
print(low_conf_sample)

            listing_id
0  1217279784872684048
1  1239163475555358770
2  1221699855812557434


In [7]:
diag = duckdb.sql(
    "SELECT * FROM read_parquet('../data/gold/nyc/diagnosis.parquet')"
).df()
sample = diag[diag["listing_id"] == 1279026320984012410]["diagnosis"].iloc[0]
print(repr(sample))

'Health score: 90/100. Your price of $167 is lower than 96% of comparable listings (median $442) — there may be room to raise it.'


In [8]:
check = duckdb.sql("""
    SELECT listing_id, room_type, stay_type, price_confidence
    FROM read_parquet('../data/gold/nyc/health_score.parquet')
    WHERE listing_id = 1221699855812557434
""").df()
print(check)

            listing_id    room_type   stay_type price_confidence
0  1221699855812557434  Shared room  short stay              low


In [9]:
# Compare what the app's merged dataframe would produce vs. the source files directly
listings_raw = duckdb.sql("""
    SELECT listing_id, room_type, stay_type, price_confidence
    FROM read_parquet('../data/gold/nyc/listings_prepared.parquet')
    WHERE listing_id = 1221699855812557434
""").df()
print("From listings_prepared.parquet:")
print(listings_raw)

health_raw = duckdb.sql("""
    SELECT listing_id, price_confidence
    FROM read_parquet('../data/gold/nyc/health_score.parquet')
    WHERE listing_id = 1221699855812557434
""").df()
print()
print("From health_score.parquet:")
print(health_raw)

From listings_prepared.parquet:
            listing_id    room_type   stay_type price_confidence
0  1221699855812557434  Shared room  short stay              low

From health_score.parquet:
            listing_id price_confidence
0  1221699855812557434              low


In [11]:
listings_cols = set(duckdb.sql(
    "DESCRIBE SELECT * FROM read_parquet('../data/gold/nyc/listings_prepared.parquet')"
).df()["column_name"])
comps_cols = set(duckdb.sql(
    "DESCRIBE SELECT * FROM read_parquet('../data/gold/nyc/comps.parquet')"
).df()["column_name"])

overlap = listings_cols & comps_cols
print("Columns present in BOTH listings_prepared and comps:", overlap)

Columns present in BOTH listings_prepared and comps: {'base_price', 'listing_id', 'price_outlier'}


In [12]:
dupe_check_listings = duckdb.sql("""
    SELECT listing_id, COUNT(*) AS n
    FROM read_parquet('../data/gold/nyc/listings_prepared.parquet')
    GROUP BY listing_id HAVING COUNT(*) > 1
""").df()
print("Duplicate listing_ids in listings_prepared:", len(dupe_check_listings))

dupe_check_comps = duckdb.sql("""
    SELECT listing_id, COUNT(*) AS n
    FROM read_parquet('../data/gold/nyc/comps.parquet')
    GROUP BY listing_id HAVING COUNT(*) > 1
""").df()
print("Duplicate listing_ids in comps:", len(dupe_check_comps))

dupe_check_health = duckdb.sql("""
    SELECT listing_id, COUNT(*) AS n
    FROM read_parquet('../data/gold/nyc/health_score.parquet')
    GROUP BY listing_id HAVING COUNT(*) > 1
""").df()
print("Duplicate listing_ids in health_score:", len(dupe_check_health))

Duplicate listing_ids in listings_prepared: 0
Duplicate listing_ids in comps: 0
Duplicate listing_ids in health_score: 0


In [13]:
import duckdb, pandas as pd

GOLD = "../data/gold/nyc"
listings_check = duckdb.sql(f"""
    SELECT * FROM read_parquet('{GOLD}/listings_prepared.parquet')
    WHERE base_price IS NOT NULL
""").df()
comps_check = duckdb.sql(f"SELECT * FROM read_parquet('{GOLD}/comps.parquet')").df()
health_check = duckdb.sql(f"SELECT * FROM read_parquet('{GOLD}/health_score.parquet')").df()

merged_check = listings_check.merge(comps_check, on="listing_id", suffixes=("", "_comp"))
merged_check = merged_check.merge(
    health_check[["listing_id", "health_score"] + [c for c in health_check.columns if c.startswith("flag_")]],
    on="listing_id"
)

print("Rows before merge:", len(listings_check))
print("Rows after merge:", len(merged_check))

result = merged_check[merged_check["listing_id"] == 1221699855812557434]
print()
print(result[["listing_id", "room_type", "stay_type"]])

Rows before merge: 21514
Rows after merge: 21514

                listing_id    room_type   stay_type
15044  1221699855812557434  Shared room  short stay


In [14]:
import duckdb
df = duckdb.sql(
    "SELECT * FROM read_parquet('../data/gold/nyc/listings_prepared.parquet')"
).df()

print(df["listing_id"].dtype)
print(df.loc[df["listing_id"].between(1221699855812557000, 1221699855812558000), "listing_id"].tolist())

int64
[1221699855812557434]


In [17]:
diag_check = duckdb.sql("""
    SELECT l.listing_id, l.neighbourhood, l.base_price, d.diagnosis
    FROM read_parquet('../data/gold/nyc/listings_prepared.parquet') l
    JOIN read_parquet('../data/gold/nyc/diagnosis.parquet') d
      ON d.listing_id = l.listing_id
    WHERE l.neighbourhood = 'Woodhaven'
      AND l.base_price BETWEEN 267 AND 269
""").df()
print(diag_check)

            listing_id neighbourhood  base_price  \
0  1356177255367743201     Woodhaven  268.243333   
1   672459717259106879     Woodhaven  268.243333   

                                                                                                                                                                                                                                                                        diagnosis  
0  Health score: 65/100. Your price of $268 is higher than 93% of comparable listings nearby (median $172) — worth checking if that's intentional. You have 2 reviews total, but none in the last 12 months — worth checking if something has changed with visibility or pricing.  
1                                                                                                                                 Health score: 80/100. Your price of $268 is higher than 93% of comparable listings nearby (median $172) — worth checking if that's intentional.  


In [18]:
print(repr(diag_check.loc[0, "diagnosis"]))

"Health score: 65/100. Your price of $268 is higher than 93% of comparable listings nearby (median $172) — worth checking if that's intentional. You have 2 reviews total, but none in the last 12 months — worth checking if something has changed with visibility or pricing."


In [1]:
import duckdb

candidates = duckdb.sql("""
    SELECT listing_id, neighbourhood, borough, room_type, base_price,
           health_score, flag_overpriced, flag_underpriced, flag_price_outlier
    FROM read_parquet('../data/gold/nyc/health_score.parquet')
    WHERE (flag_overpriced OR flag_underpriced)
      AND price_confidence = 'normal'
    ORDER BY health_score
    LIMIT 20
""").df()
print(candidates)

             listing_id       neighbourhood    borough        room_type  \
0              49920227            Longwood      Bronx     Private room   
1              39553889    Long Island City     Queens     Private room   
2               1623431             Midtown  Manhattan     Private room   
3                990529         Murray Hill  Manhattan     Private room   
4              25134894           Chinatown  Manhattan     Private room   
5              25519328              Harlem  Manhattan      Shared room   
6    810830041327117144    Prospect Heights   Brooklyn     Private room   
7    823919993371756124       Cypress Hills   Brooklyn     Private room   
8              48569389             Chelsea  Manhattan     Private room   
9   1041449989786816094             Chelsea  Manhattan  Entire home/apt   
10             39554366    Long Island City     Queens     Private room   
11             39554839    Long Island City     Queens     Private room   
12   877469016958154685  

In [2]:
detail = duckdb.sql("""
    SELECT h.listing_id, h.base_price, h.pct_comps_cheaper, h.comp_median,
           h.availability_30, h.avg_availability_30, d.diagnosis
    FROM read_parquet('../data/gold/nyc/health_score.parquet') h
    JOIN read_parquet('../data/gold/nyc/diagnosis.parquet') d
      ON d.listing_id = h.listing_id
    WHERE h.listing_id = 775404417497255085
""").df()
print(detail.to_string())

           listing_id  base_price  pct_comps_cheaper  comp_median  availability_30  avg_availability_30                                                                                                                                                                                                                                                                                                                                                                                                                  diagnosis
0  775404417497255085       546.0          89.655172        351.9               28            13.253933  Health score: 20/100. Your price of $546 is higher than 90% of comparable listings nearby (median $352) — worth checking if that's intentional. Your calendar is 93% open in the next 30 days, versus 44% for similar listings — this could mean the price or listing needs a closer look. You have 1 reviews total, but none in the last 12 months — worth checking if something has change

In [3]:
underpriced_candidates = duckdb.sql("""
    SELECT h.listing_id, h.neighbourhood, h.borough, h.room_type, h.base_price,
           h.health_score, h.pct_comps_cheaper, h.comp_median, d.diagnosis
    FROM read_parquet('../data/gold/nyc/health_score.parquet') h
    JOIN read_parquet('../data/gold/nyc/diagnosis.parquet') d
      ON d.listing_id = h.listing_id
    WHERE h.flag_underpriced = TRUE
      AND h.price_confidence = 'normal'
    ORDER BY h.base_price DESC
    LIMIT 10
""").df()
print(underpriced_candidates.to_string())

            listing_id       neighbourhood    borough        room_type  base_price  health_score  pct_comps_cheaper  comp_median                                                                                                                                                                                                                                                                      diagnosis
0  1111672634422819489     Upper East Side  Manhattan       Hotel room  932.000000            90           9.523810  1632.000000                                                                                                                                             Health score: 90/100. Your price of $932 is lower than 90% of comparable listings (median $1,632) — there may be room to raise it.
1  1111671446835633847     Upper East Side  Manhattan       Hotel room  932.000000            90           9.523810  1632.000000                                                                        

In [4]:
import duckdb

impact = duckdb.sql("""
    SELECT
        COUNT(*) FILTER (WHERE flag_underpriced) AS underpriced_listings,
        COUNT(*) FILTER (WHERE flag_overpriced) AS overpriced_listings,
        ROUND(AVG(comp_median - base_price) FILTER (WHERE flag_underpriced), 0) AS avg_dollars_left_on_table,
        ROUND(100.0 * COUNT(*) FILTER (WHERE flag_underpriced OR flag_overpriced) / COUNT(*), 1) AS pct_listings_mispriced,
        COUNT(*) FILTER (WHERE health_score < 50) AS listings_below_50,
        ROUND(100.0 * COUNT(*) FILTER (WHERE health_score < 50) / COUNT(*), 1) AS pct_below_50
    FROM read_parquet('../data/gold/nyc/health_score.parquet')
""").df()
print(impact.T)

                                0
underpriced_listings       5533.0
overpriced_listings        6102.0
avg_dollars_left_on_table    89.0
pct_listings_mispriced       54.1
listings_below_50          1506.0
pct_below_50                  7.0
